Propósito: Filtra los tickers propuestos para traerlos filtrados por condiciones.

Usa parametros del indicador RSI de 60min, 1d y 7d.

Este filtro busca capturar movimientos de la siguiente forma: Activo sobrevendido en 1d y 60m; la temporalidad baja indica el movimiento de reversion alcista en conjunto con una temporalidad diaria subiendo. La temporalidad más larga indica la tendencia del activo en largo plazo.

In [ ]:
# Instala las librerias requeridas.

%pip install -q pandas-ta yfinance lxml

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Importa las librerias requeridas.
import pandas as pd
import requests
import io
import yfinance as yf
import pandas_ta as ta

In [11]:
#Scrapea Wikipedia para acceder a los nombres de los tickers del S&P500.

#1.1 Define la URL y un 'User-Agent' para evitar el bloqueo.
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
header = {
  "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/50.0.2661.75 Safari/537.36",
  "X-Requested-With": "XMLHttpRequest"
}

#1.2 Realiza la petición con el header de seguridad
r = requests.get(url, headers=header)

#1.3 Lee la tabla desde el texto de la respuesta, usando io.StringIO para forzar la interpretación como string HTML
tables = pd.read_html(io.StringIO(r.text))
df_spy = tables[0]

#1.4 Extrae la lista de Tickers
tickers_a_analizar = [t.replace('.', '-') for t in df_spy['Symbol'].tolist()]

#1.5 Añade tickers de interes.

mis_tickers = ['^VIX', 'SPY', 'SPXL' , 'TQQQ' , 'QQQ' , 'URA' , 'XLI' , 'XLF' , 'XLRE' , 'XLB' , 'XLY' , 'XLE', 'XLV' ,
                'XBI' , 'XLC' , 'BTC-USD' , 'DIA']

tickers_a_analizar.extend(mis_tickers)

if tickers_a_analizar:
  print(f"Se han cargado {len(tickers_a_analizar)} tickers...")

Se han cargado 520 tickers...


In [ ]:
#DESCARGA DE DATOS TEMPORALIDAD 60 MINUTOS.

tickers_limpios = tickers_a_analizar
data_60m = yf.download(tickers_limpios, period="2mo", interval="60m", group_by='ticker', progress=False)

Descargando datos para 520 activos...


In [5]:
#ANALISIS TEMPORALIDAD 60MIN.

empresas_60min = []

for ticker in tickers_limpios:
    try:
        df = data_60m[ticker].dropna()
        if len(df) < 50: continue

        # Cálculos Técnicos
        df.ta.rsi(length=10, append=True)
        df['RSI_MA'] = ta.sma(df['RSI_10'], length=14)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        # Variables para validación.
        rsi_act = df['RSI_10'].iloc[-1]
        rsi_ant = df['RSI_10'].iloc[-2]
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]

        # Lógica de Filtrado 60min
        if (rsi_act <= 35 
            and rsi_act >= rsi_ant
                ):
            
            empresas_60min.append({
                'Ticker': ticker,
                'RSI actual': round(rsi_act, 1),
                'RSI anterior': round(rsi_ant, 1),
                #Añade el MACD.
                'MACD': round(macd_act, 3)
                
            })
    except Exception as e:
      print(f"Error en {ticker}: {e}")

print("\n" + "="*95)
if empresas_60min:
    df_final = pd.DataFrame(empresas_60min)
    print(df_final.sort_values(by='RSI actual').to_string(index=False))
else:
    print("Sin señales detectadas.")


Ticker  RSI actual  RSI anterior    MACD
    GM        22.2          15.3  -0.777
   PWR        24.3          23.5 -12.500
   ADM        27.2          24.4  -0.588
   VLO        27.8          16.1  -3.981
   CEG        28.8          22.8  -5.031
   GPC        29.2          18.1  -0.597
   EME        30.6          29.1  -9.197


In [12]:
# DESCARGA DATOS DIARIOS
data_1d = yf.download(tickers_limpios, period="2mo", interval="1d", group_by='ticker', progress=False)

In [7]:
# ANÁLISIS DIARIO

empresas_diario = []

for ticker in tickers_limpios:
    try:
        df = data_1d[ticker].dropna()
        if len(df) < 30: continue

        # Cálculos Técnicos.
        df.ta.rsi(length=14, append=True)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        # Variables de validación
        rsi_ant = df['RSI_14'].iloc[-2]
        rsi_act = df['RSI_14'].iloc[-1]
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]
        

        precio_act = df['Close'].iloc[-1]

        # LÓGICA DE FILTRADO DIARIO
        estuvo_sobrevendido = rsi_act <= 35 or rsi_ant <= 35
        dejo_de_caer = rsi_act >= rsi_ant

        if estuvo_sobrevendido:
            
            empresas_diario.append({
                'Ticker': ticker,
                #'Prec diario': round(precio_act, 2),
                'RSI actual': round(rsi_act, 1),
                'RSI anterior': round(rsi_ant, 1),
                'MACD': round(macd_act, 3),
                #'MACD_SIGNAL': round(sig_act , 3)
            })
    except:
        continue

print("\n" + "="*95)


print("PAPELES DETECTADOS EN DIARIO:")
if empresas_diario:
    df_final = pd.DataFrame(empresas_diario)
    print(df_final.sort_values(by='RSI actual').to_string(index=False))
else:
    print("Sin señales diarias.")


PAPELES DETECTADOS EN DIARIO:
Ticker  RSI actual  RSI anterior    MACD
   MGM        24.4          21.0  -2.706
   VRT        26.3          25.3 -11.867
   BAC        27.3          20.3  -2.216
  AMAT        28.3          29.2 -13.806
   MOS        28.8          29.1  -1.535
  KLAC        28.9          30.4  -3.761
  FLEX        30.1          27.6  -4.771
   ARE        31.0          25.1  -2.442
  SKYD        31.0          25.9  -0.871
   FIX        31.3          33.4 -40.423
   HWM        31.5          36.9  -6.969
    BG        33.3          33.5  -2.816
   STX        33.4          30.3 -47.485
  SMCI        33.5          32.3  -1.507
  COHR        33.9          32.2 -13.344
  LRCX        34.0          35.6  -9.030
   PWR        34.0          35.9  -0.352
   ETN        34.0          36.1  -5.448
   EME        34.4          35.7  -8.539
   CAT        34.7          32.5 -20.475
   WDC        34.8          32.1 -26.033
  AKAM        34.9          29.9  -4.766
  INTC        35.6        

In [27]:
#TICKERS REPETIDOS EN AMBAS TEMPORALIDADES.

tickers_60 = [e['Ticker'] for e in empresas_60min]
ticker_1d = [e['Ticker'] for e in empresas_diario]
tickers_comunes = list(set(tickers_60) & set(ticker_1d)) # Interseccion de los elementos en la lista.

print(tickers_comunes)


['PWR', 'EME']


In [15]:
# DESCARGA DATOS SEMANAL.
data_7d = yf.download(tickers_limpios, period="1y", interval="1wk", group_by='ticker', progress=False)

In [16]:
# ANÁLISIS SEMANAL

empresas_semanal = []

for ticker in tickers_limpios:
    try:
        df = data_7d[ticker].dropna()
        if len(df) < 50: continue

        # Cálculos Técnicos.
        df.ta.rsi(length=10, append=True)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        # Variables de validación
        rsi_act = df['RSI_10'].iloc[-1]
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]
        rsi_ant = df['RSI_10'].iloc[-2]

        # Lógica de filtrado.
        if rsi_act <= 44 and (rsi_act > rsi_ant):
        
            empresas_semanal.append({
                'Ticker': ticker,
                'RSI actual': round(rsi_act, 1),
                'RSI anterior': round(rsi_ant, 1),
                'MACD': round(macd_act, 3),
            })
    except:
        continue

print("\n" + "="*95)
print("ACTIVOS DETECTADOS EN SEMANAL.")
if empresas_semanal:
    df_final = pd.DataFrame(empresas_semanal)
    print(df_final.sort_values(by='RSI actual').to_string(index=False))
else:
    print("Sin señales semanales.")


ACTIVOS DETECTADOS EN SEMANAL.
Ticker  RSI actual  RSI anterior     MACD
   MGM        16.5          16.1   -2.806
  WYNN        26.6          21.3   -8.014
   URA        27.9          27.4   -2.631
   PLD        28.4          28.1   -2.097
  SKYD        28.4          26.1   -0.721
    MS        29.2          27.5   -1.138
   LVS        30.5          29.8   -4.132
    GS        31.2          27.2  -11.450
   LII        31.2          28.8  -39.913
   APP        32.4          28.9  -60.595
   PCG        32.6          30.5   -1.012
   CRH        32.7          29.3   -7.288
  APTV        33.0          31.7   -6.097
   CMI        34.4          32.9  -20.378
  AKAM        34.6          32.7   -3.968
   BXP        34.9          34.1   -0.774
  LULU        35.7          32.8  -13.552
   MAA        36.1          34.0   -3.398
  TMUS        36.5          36.2   -8.640
   PNC        36.5          33.2    0.151
  AXON        36.6          34.7  -25.243
   TXT        36.6          34.5   -3.785
  

In [28]:
#TICKERS REPETIDOS EN TRIPLE PANTALLA.

tickers_60 = [e['Ticker'] for e in empresas_60min]
tickers_1d = [e['Ticker'] for e in empresas_diario]
tickers_7d = [e['Ticker'] for e in empresas_semanal]

tickers_comunes = list(set(tickers_60) & set(tickers_1d) & set(tickers_7d))

if tickers_comunes:
    print(tickers_comunes)